# MLflow Tracing & AI Gateway Demo

This notebook demonstrates:
1. **MLflow Tracing** - Automatic instrumentation of ML training and inference
2. **AI Gateway** - Unified access to Claude and Ollama models
3. **Governance Dashboard** - View traces in MLflow UI

## Architecture
```
Jupyter Notebook
    |
    +-- MLflow Tracing (auto-logs training/inference)
    |       |
    |       v
    |   MLflow Server (http://mlflow:5000)
    |       |
    |       +-- PostgreSQL (experiment metadata)
    |       +-- MinIO S3 (artifacts)
    |
    +-- AI Gateway (LLM access)
            |
            +-- Claude via Anthropic
            +-- Local models via Ollama
```

**MLflow UI**: http://localhost:5001

## 1. Setup and Configuration

In [ ]:
# Install required packages (boto3 needed for S3/MinIO artifact storage)
!pip install -q mlflow xgboost scikit-learn openai anthropic ollama boto3

In [ ]:
import os

# Disable inline trace viewer BEFORE importing mlflow (browser can't resolve 'mlflow' hostname)
# Traces are still logged - view them at http://localhost:5001
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import numpy as np
import pandas as pd
from datetime import datetime

# Configure MLflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT_NAME", "fraud-detection")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {MLFLOW_EXPERIMENT}")
print(f"MLflow Version: {mlflow.__version__}")
print(f"\nTraces are logged to MLflow. View at: http://localhost:5001")

MLflow Tracking URI: http://mlflow:5000
Experiment: fraud-detection
MLflow Version: 3.11.1

Traces are logged to MLflow. View at: http://localhost:5001


## 2. MLflow Tracing for ML Training

MLflow 3.0 provides automatic tracing with `mlflow.autolog()`. This captures:
- Model parameters
- Training metrics
- Model artifacts
- Feature importance
- Training duration

In [ ]:
# Enable auto-logging for XGBoost
mlflow.xgboost.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True
)

print("XGBoost auto-logging enabled")

XGBoost auto-logging enabled


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Generate synthetic fraud data for demo
np.random.seed(42)
n_samples = 10000

# Features: amount, hour, day_of_week, merchant_risk, channel_risk
X = np.random.randn(n_samples, 5)
X[:, 0] = np.abs(X[:, 0]) * 500  # amount
X[:, 1] = np.random.randint(0, 24, n_samples)  # hour
X[:, 2] = np.random.randint(1, 8, n_samples)  # day_of_week

# Fraud label (2% fraud rate with pattern)
y = ((X[:, 0] > 400) & (X[:, 1] > 22) | (X[:, 3] > 2)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Fraud rate: {y.mean()*100:.2f}%")

Training samples: 8000
Test samples: 2000
Fraud rate: 4.03%


In [ ]:
# Train with MLflow tracing
with mlflow.start_run(run_name=f"fraud-training-{datetime.now().strftime('%H%M%S')}") as run:
    # Log custom tags
    mlflow.set_tags({
        "model_type": "xgboost",
        "use_case": "fraud_detection",
        "data_source": "synthetic"
    })
    
    # Train XGBoost model (auto-logged by mlflow.xgboost.autolog)
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    # Evaluate
    y_pred = model.predict(X_test)
    
    # Log additional metrics
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })
    
    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
    print(f"\nView in MLflow UI: http://localhost:5001/#/experiments")

2026/04/09 13:12:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 4cc1f38f9dfb4811b1f3e2a7cbfc7d38
Accuracy: 0.9970
F1 Score: 0.9630

View in MLflow UI: http://localhost:5001/#/experiments
🏃 View run fraud-training-131247 at: http://mlflow:5000/#/experiments/1/runs/4cc1f38f9dfb4811b1f3e2a7cbfc7d38
🧪 View experiment at: http://mlflow:5000/#/experiments/1


## 3. MLflow Tracing for Inference

Use the `@mlflow.trace` decorator to instrument inference functions.
This captures:
- Input features
- Predictions
- Latency
- Any errors

In [ ]:
@mlflow.trace(name="fraud_prediction", span_type="FUNCTION")
def predict_fraud(features: dict) -> dict:
    """
    Predict fraud probability for a transaction.
    Traced by MLflow for observability.
    """
    # Convert to numpy array
    X = np.array([[
        features.get("amount", 0),
        features.get("hour", 12),
        features.get("day_of_week", 1),
        features.get("merchant_risk", 0),
        features.get("channel_risk", 0)
    ]])
    
    # Predict
    proba = model.predict_proba(X)[0]
    is_fraud = proba[1] > 0.5
    
    return {
        "is_fraud": bool(is_fraud),
        "fraud_probability": float(proba[1]),
        "safe_probability": float(proba[0])
    }

print("Inference function defined with MLflow tracing")

Inference function defined with MLflow tracing


In [ ]:
# Test traced inference
test_transactions = [
    {"amount": 50, "hour": 10, "day_of_week": 2, "merchant_risk": 0.1, "channel_risk": 0.1},
    {"amount": 500, "hour": 23, "day_of_week": 7, "merchant_risk": 2.5, "channel_risk": 0.5},
    {"amount": 1500, "hour": 3, "day_of_week": 1, "merchant_risk": 3.0, "channel_risk": 2.0},
]

print("Running traced inference...\n")
for i, tx in enumerate(test_transactions, 1):
    result = predict_fraud(tx)
    status = "FRAUD" if result["is_fraud"] else "SAFE"
    print(f"Transaction {i}: ${tx['amount']:.0f} at hour {tx['hour']} -> {status} ({result['fraud_probability']:.2%})")

print(f"\nView traces in MLflow UI: http://localhost:5001")

Running traced inference...

Transaction 1: $50 at hour 10 -> SAFE (0.01%)
Transaction 2: $500 at hour 23 -> FRAUD (99.64%)
Transaction 3: $1500 at hour 3 -> FRAUD (99.19%)

View traces in MLflow UI: http://localhost:5001


[Trace(trace_id=tr-9f73dce331df4406c46369dbf65577e8), Trace(trace_id=tr-790a111d02b5a49e7b6ce452ac96e4c3), Trace(trace_id=tr-0db7f7c869a56e26f1a484f5495888e7)]

## 4. AI Gateway - Unified LLM Access

MLflow AI Gateway provides:
- **Unified API**: Same interface for Claude, Ollama, OpenAI, etc.
- **Centralized credentials**: API keys stored securely in MLflow
- **Tracing**: All LLM calls automatically logged
- **Governance**: Rate limiting, access control, audit logs

In [ ]:
# MLflow 3.0 AI Gateway Status
# Note: MLflow 3.0 configures gateway routes via UI, not REST API
# Configure at: http://localhost:5001 -> Settings -> AI Gateway

import requests

GATEWAY_URI = os.getenv("MLFLOW_GATEWAY_URI", "http://mlflow:5000/gateway")

print("MLflow AI Gateway")
print("=" * 50)
print("MLflow 3.0 uses UI-based gateway configuration.")
print("Configure endpoints at: http://localhost:5001")
print("  1. Click Settings (gear icon)")
print("  2. Select 'AI Gateway'")
print("  3. Add API keys and create endpoints")
print("")
print("For this demo, we'll use Ollama directly (no gateway needed).")

MLflow AI Gateway
MLflow 3.0 uses UI-based gateway configuration.
Configure endpoints at: http://localhost:5001
  1. Click Settings (gear icon)
  2. Select 'AI Gateway'
  3. Add API keys and create endpoints

For this demo, we'll use Ollama directly (no gateway needed).


### 4.1 Query Ollama (Local Models)

In [ ]:
# Direct Ollama query (for comparison)
import ollama

OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://host.docker.internal:11434")

try:
    # List available models
    client = ollama.Client(host=OLLAMA_URL)
    response = client.list()
    
    print("Available Ollama Models:")
    print("=" * 50)
    
    # Handle different response formats (dict or object)
    models = response.get("models", []) if isinstance(response, dict) else getattr(response, 'models', [])
    
    for model in models:
        # Try different attribute/key names
        if isinstance(model, dict):
            name = model.get('name') or model.get('model') or str(model)
        else:
            name = getattr(model, 'name', None) or getattr(model, 'model', str(model))
        print(f"  - {name}")
        
    if not models:
        print("  No models found. Pull a model with: ollama pull llama3.2")
        
except Exception as e:
    print(f"Ollama not available: {e}")
    print("\nMake sure Ollama is running on your host machine.")
    print("Install: https://ollama.ai")
    print("Pull a model: ollama pull llama3.2")

Available Ollama Models:
  - nomic-embed-text:latest
  - gpt-oss:20b
  - gpt-oss:latest


In [ ]:
@mlflow.trace(name="ollama_fraud_analysis", span_type="LLM")
def analyze_fraud_with_ollama(transaction: dict, model: str = "gpt-oss:20b") -> str:
    """
    Use local Ollama model to analyze a suspicious transaction.
    Traced by MLflow for governance.
    """
    prompt = f"""Analyze this transaction for potential fraud indicators:

Transaction Details:
- Amount: ${transaction.get('amount', 0)}
- Time: {transaction.get('hour', 12)}:00
- Day: {['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][transaction.get('day_of_week', 0) - 1]}
- Merchant Risk Score: {transaction.get('merchant_risk', 0)}
- Channel Risk Score: {transaction.get('channel_risk', 0)}

Provide a brief risk assessment (2-3 sentences)."""

    try:
        client = ollama.Client(host=OLLAMA_URL)
        response = client.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response["message"]["content"]
    except Exception as e:
        return f"Analysis unavailable: {e}"

# Test with a suspicious transaction
suspicious_tx = {
    "amount": 1500,
    "hour": 3,
    "day_of_week": 7,
    "merchant_risk": 3.0,
    "channel_risk": 2.0
}

print("Analyzing suspicious transaction with Ollama (gpt-oss:20b)...\n")
analysis = analyze_fraud_with_ollama(suspicious_tx)
print(f"Analysis:\n{analysis}")

### 4.2 Query Claude via AI Gateway


In [ ]:
import anthropic

@mlflow.trace(name="claude_fraud_analysis", span_type="LLM")
def analyze_fraud_with_claude(transaction: dict) -> str:
    """
    Use Claude (via Anthropic SDK) to analyze a suspicious transaction.
    Traced by MLflow for governance.
    """
    prompt = f"""Analyze this transaction for potential fraud indicators:

Transaction Details:
- Amount: ${transaction.get('amount', 0)}
- Time: {transaction.get('hour', 12)}:00
- Day: {['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][transaction.get('day_of_week', 0) - 1]}
- Merchant Risk Score: {transaction.get('merchant_risk', 0)}
- Channel Risk Score: {transaction.get('channel_risk', 0)}

Provide a brief risk assessment (2-3 sentences)."""

    # Try direct Anthropic API first
    api_key = os.getenv("ANTHROPIC_API_KEY", "")
    model = os.getenv("ANTHROPIC_DEFAULT_MODEL", "claude-4-6")
    
    if api_key:
        try:
            client = anthropic.Anthropic(api_key=api_key)
            response = client.messages.create(
                model=model,
                max_tokens=200,
                messages=[{"role": "user", "content": prompt}]
            )
            return "".join([block.text for block in response.content if block.type == "text"])
        except Exception as e:
            return f"Claude API error: {e}"
    else:
        return "Claude unavailable: Set ANTHROPIC_API_KEY env var, or configure via MLflow UI Gateway"

# Test with suspicious transaction
print("Analyzing suspicious transaction with Claude...\n")
claude_analysis = analyze_fraud_with_claude(suspicious_tx)
print(f"Analysis:\n{claude_analysis}")

## 5. Combined ML + LLM Pipeline with Tracing

This demonstrates a complete fraud detection pipeline:
1. ML model predicts fraud probability
2. If suspicious, LLM provides detailed analysis
3. All steps traced in MLflow

In [ ]:
@mlflow.trace(name="fraud_detection_pipeline", span_type="CHAIN")
def detect_fraud_pipeline(transaction: dict, use_llm: str = "ollama") -> dict:
    """
    Complete fraud detection pipeline with ML + LLM.
    Fully traced in MLflow.
    
    Args:
        transaction: Transaction features
        use_llm: "ollama" for local, "claude" for Anthropic
    
    Returns:
        Detection result with ML prediction and LLM analysis
    """
    # Step 1: ML Prediction
    ml_result = predict_fraud(transaction)
    
    result = {
        "transaction": transaction,
        "ml_prediction": ml_result,
        "llm_analysis": None,
        "final_decision": "SAFE"
    }
    
    # Step 2: If suspicious (>30% fraud probability), get LLM analysis
    if ml_result["fraud_probability"] > 0.3:
        if use_llm == "claude":
            result["llm_analysis"] = analyze_fraud_with_claude(transaction)
        else:
            result["llm_analysis"] = analyze_fraud_with_ollama(transaction)
        
        result["final_decision"] = "REVIEW" if ml_result["fraud_probability"] > 0.5 else "MONITOR"
    
    if ml_result["fraud_probability"] > 0.7:
        result["final_decision"] = "BLOCK"
    
    return result

print("Fraud detection pipeline defined")

In [ ]:
# Run pipeline on test transactions
test_cases = [
    {"amount": 25, "hour": 14, "day_of_week": 3, "merchant_risk": 0.1, "channel_risk": 0.1},
    {"amount": 350, "hour": 20, "day_of_week": 5, "merchant_risk": 1.5, "channel_risk": 0.8},
    {"amount": 2500, "hour": 2, "day_of_week": 7, "merchant_risk": 3.5, "channel_risk": 2.5},
]

print("Running Fraud Detection Pipeline")
print("=" * 70)

for i, tx in enumerate(test_cases, 1):
    print(f"\nTransaction {i}: ${tx['amount']} at {tx['hour']}:00")
    print("-" * 50)
    
    result = detect_fraud_pipeline(tx, use_llm="ollama")
    
    print(f"ML Prediction: {result['ml_prediction']['fraud_probability']:.2%} fraud probability")
    print(f"Final Decision: {result['final_decision']}")
    
    if result['llm_analysis']:
        print(f"LLM Analysis: {result['llm_analysis'][:200]}...")

print("\n" + "=" * 70)
print("View all traces in MLflow UI: http://localhost:5001")

## 6. View Traces in MLflow UI

All the operations above are traced and visible in MLflow:

1. **Open MLflow UI**: http://localhost:5001
2. **Navigate to Experiments** tab
3. **Select "fraud-detection"** experiment
4. **Click on a run** to see:
   - Training parameters and metrics
   - Model artifacts
   - Traces tab for inference and LLM calls

### Trace Dashboard Features:
- **Latency breakdown** per operation
- **Input/output** for each traced function
- **Error tracking** with stack traces
- **LLM token usage** and costs (when available)
- **Nested spans** for complex pipelines

In [ ]:
# List recent runs and traces
print("Recent MLflow Runs:")
print("=" * 70)

runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    max_results=5,
    order_by=["start_time DESC"]
)

if not runs.empty:
    for _, run in runs.iterrows():
        print(f"\nRun: {run['run_id'][:8]}...")
        print(f"  Name: {run.get('tags.mlflow.runName', 'N/A')}")
        print(f"  Status: {run['status']}")
        print(f"  Start: {run['start_time']}")
        
        # Show metrics if available
        metrics = [col for col in run.index if col.startswith('metrics.')]
        if metrics:
            print(f"  Metrics: {', '.join([f'{m.split(".")[1]}={run[m]:.4f}' for m in metrics[:3] if pd.notna(run[m])])}")
else:
    print("No runs found. Run the training cells above first.")

print(f"\nView all in MLflow UI: http://localhost:5001/#/experiments")

## 7. Summary

### What We Covered:

1. **MLflow Tracing Setup**
   - `mlflow.set_tracking_uri()` - Connect to MLflow server
   - `mlflow.xgboost.autolog()` - Automatic XGBoost instrumentation
   - `@mlflow.trace` decorator - Custom function tracing

2. **AI Gateway Configuration**
   - Claude via Anthropic (enterprise)
   - Ollama for local models (development)
   - OpenAI-compatible API format

3. **Combined ML + LLM Pipeline**
   - ML model for initial prediction
   - LLM for detailed analysis of suspicious cases
   - Full tracing for governance

### Configuration Variables:
```bash
# MLflow
MLFLOW_TRACKING_URI=http://mlflow:5000
MLFLOW_EXPERIMENT_NAME=fraud-detection

# Anthropic (for Claude)
ANTHROPIC_API_KEY=your-api-key

# Ollama (local)
OLLAMA_BASE_URL=http://host.docker.internal:11434
```

### Next Steps:
- Explore the MLflow UI at http://localhost:5001
- Add more LLM models via AI Gateway
- Set up alerts for suspicious patterns
- Export traces for compliance reporting